Import and shared constants

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
(ROOT / "results").mkdir(exist_ok=True)
(ROOT / "figures").mkdir(exist_ok=True)

SEED = 42
R_SERVICE_KM = 1.0      # service radius of a station
D_MIN_KM = 1.0          # minimum distance between two stations
MAX_NEW_STATIONS = 10   # maximum number of new stations
GRID_STEP_KM = 1.0      # grid step size for candidate generation
COST_PER_STATION = 1.0  # unit cost per station
COST_PER_KM = 2.0       # unit cost per kilometer of route
BUDGET = 40.0           # total budget
DEFAULT_WEIGHTS = {"pop": 1.0, "cost": 0.3, "penalty": 5.0}

---

# STUB TẠM — Khối 1, 2, 3 (phần của TV1)

Cell dưới là bản tạm để Khối 4 chạy thử được ngay, **không phải code chính thức**.
Khi TV1 nộp Khối 1, 2, 3 thì xoá hẳn cell này đi.

Nó dựng đúng 5 thứ mà Khối 4 cần:

| Tên | Là gì |
|---|---|
| `haversine` | Khoảng cách km giữa hai toạ độ |
| `randomState` | Bốc đại một bộ vị trí đặt trạm |
| `objective` | Chấm điểm một phương án |
| `getNeighbors` | Sinh mọi phương án chỉnh nhẹ 1 trạm |
| `randomNeighbor` | Bốc đại một phương án chỉnh nhẹ |

Vị trí ứng viên tạm lấy luôn tâm 168 phường trong `data/wards.csv`.
TV1 sẽ thay bằng lưới điểm phủ khu vực theo `GRID_STEP_KM`.

In [2]:
# ====== STUB TẠM, XOÁ KHI TV1 NỘP KHỐI 1-3 ======

wards = pd.read_csv(ROOT / "data" / "wards.csv")


def haversine(lat1, lon1, lat2, lon2):
    """Khoảng cách km giữa hai điểm trên mặt đất."""
    R = 6371.0
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dPhi = phi2 - phi1
    dLambda = np.radians(lon2) - np.radians(lon1)
    a = np.sin(dPhi / 2) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dLambda / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))


# Ứng viên tạm: tâm mỗi phường là một chỗ có thể đặt trạm
candLat = wards["lat"].to_numpy(dtype=float)
candLon = wards["lon"].to_numpy(dtype=float)
wardPop = wards["dan_so"].to_numpy(dtype=float)
totalPop = wardPop.sum()
numCandidates = len(wards)
MIN_NEW_STATIONS = 4

# Tính trước khoảng cách để chấm điểm cho nhanh, khỏi tính đi tính lại
DIST_CAND_WARD = haversine(candLat[:, None], candLon[:, None],
                           candLat[None, :], candLon[None, :])
DIST_CAND_CAND = DIST_CAND_WARD


def randomState(numStations=None, rng=None):
    """Bốc đại một bộ vị trí đặt trạm hợp lệ."""
    rng = rng if rng is not None else np.random.default_rng()
    n = numStations or int(rng.integers(MIN_NEW_STATIONS, MAX_NEW_STATIONS + 1))
    return tuple(sorted(rng.choice(numCandidates, size=n, replace=False).tolist()))


def routeLengthKm(idx):
    """Chiều dài tuyến nối các trạm, xấp xỉ bằng cách nối lần lượt từ tây sang đông."""
    if len(idx) < 2:
        return 0.0
    order = sorted(idx, key=lambda c: (candLon[c], candLat[c]))
    return float(sum(DIST_CAND_CAND[order[i], order[i + 1]] for i in range(len(order) - 1)))


def objective(state, weights=None):
    """Chấm điểm một phương án: dân phục vụ được, trừ chi phí, trừ phạt vi phạm."""
    w = weights if weights is not None else DEFAULT_WEIGHTS
    idx = list(state)

    # Phường được phục vụ nếu nằm trong bán kính của ít nhất một trạm
    served = (DIST_CAND_WARD[idx] <= R_SERVICE_KM).any(axis=0)
    popScore = wardPop[served].sum() / totalPop

    cost = len(idx) * COST_PER_STATION + routeLengthKm(idx) * COST_PER_KM

    penalty = 0.0
    if len(idx) > 1:
        sub = DIST_CAND_CAND[np.ix_(idx, idx)]
        tooClose = ((sub < D_MIN_KM).sum() - len(idx)) / 2
        penalty += tooClose
    if len(idx) > MAX_NEW_STATIONS:
        penalty += len(idx) - MAX_NEW_STATIONS
    if cost > BUDGET:
        penalty += (cost - BUDGET) / BUDGET

    return w["pop"] * popScore - w["cost"] * (cost / BUDGET) - w["penalty"] * penalty


def getNeighbors(state):
    """Mọi phương án chỉnh nhẹ 1 trạm: dời sang chỗ khác, mở thêm, hoặc bỏ bớt."""
    chosen = set(state)
    unchosen = [c for c in range(numCandidates) if c not in chosen]
    out = []
    for old in state:
        for new in unchosen:
            out.append(tuple(sorted(chosen - {old} | {new})))
    if len(state) < MAX_NEW_STATIONS:
        out += [tuple(sorted(chosen | {new})) for new in unchosen]
    if len(state) > MIN_NEW_STATIONS:
        out += [tuple(sorted(chosen - {old})) for old in state]
    return out


def randomNeighbor(state, rng):
    """Bốc đại một phương án chỉnh nhẹ, dùng cho stochastic beam."""
    pool = getNeighbors(state)
    return pool[int(rng.integers(len(pool)))]


print(f"Stub san sang: {numCandidates} ung vien, {totalPop:,.0f} dan")

Stub san sang: 168 ung vien, 13,803,797 dan


## 4.1 localBeamSearch

Giữ beamWidth phương án song song. Mỗi vòng chỉnh nhẹ tất cả phương án
đang có, gom chung một rổ, rồi giữ lại beamWidth phương án điểm cao nhất.

beamWidth = 1 chính là Hill Climbing.


In [3]:
def localBeamSearch(beamWidth,maxRounds=100,maxNoGain=10, seed=0,numStations=None, weights=None):
    rng = np.random.default_rng(seed)
    # tạo phương án ngẫu nhiên làm điểm xuất phát
    beam = [randomState(numStations,rng) for _ in range(beamWidth)]
    beamScores = np.array([objective(state,weights) for state in beam])
    
    # ghi dữ liệu ban đầu
    bestIndex = int(np.argmax(beamScores))
    bestState = beam[bestIndex]
    bestScore = float(beamScores[bestIndex])
    
    history = [{
        "round": 0,
        "bestScore": bestScore,
        "beamBest": float(beamScores.max()),
        "beamMean": float(beamScores.mean()),
    }]
    noGainRounds = 0
    
    # với phương án đang có, chỉnh nhẹ 1 trạm dời sang trạm khác 
    for roundNo in range(1,maxRounds+1):
        neighborPool = list({
            newState
            for state in beam
            for newState in getNeighbors(state)
        })
        
        if not neighborPool:
            break
        
        neighborScores = np.array([objective(state,weights) for state in neighborPool])
        topIndexs = np.argsort(neighborScores)[::-1][:beamWidth]
        beam = [neighborPool[i] for i in topIndexs]
        beamScores = neighborScores[topIndexs]
        
        # beamScores[0] tốt nhất của riêng vòng này
        # bestScore tốt nhất từ đầu tới giờ
        if beamScores[0] > bestScore:
            bestState = beam[0]
            bestScore = float(beamScores[0])
            noGainRounds = 0
        else:
            noGainRounds +=1 
        history.append({
            "round": roundNo,
            "bestScore": bestScore,
            "beamBest": float(beamScores[0]),
            "beamMean": float(beamScores.mean()),
        })
        
        # không được thì dừng lại việc thực thi
        if noGainRounds >= maxNoGain:
            break
        
    return bestState, bestScore, history
    


## 4.2 stochasticLocalBeamSearch

Lấy top-k tuyệt đối thì cả `beamWidth` suất thường rơi hết vào con cháu của
cùng một phương án. Beam mất đa dạng, `beamWidth` phương án gần như giống hệt
nhau, thuật toán thoái hoá thành Hill Climbing.

Bốc thăm thì phương án điểm thấp vẫn có cơ hội sống sót. Một cách bố trí trạm
tạm thời kém, sau vài vòng chỉnh sửa, có thể vượt lên dẫn đầu.

In [4]:
def stochasticLocalBeamSearch(beamWidth, maxRounds=100, maxNoGain=10, seed=0,
                              numStations=None, weights=None):
    rng = np.random.default_rng(seed)
    
    # bốc đại 1 beamWidth phương án để có chỗ bắt đầu
    beam = [randomState(numStations,rng) for _ in range(beamWidth)]
    beamScores = np.array([objective(state,weights) for state in beam])
    
    bestIndex = int(np.argmax(beamScores))
    bestState = beam[bestIndex]
    bestScore = float(beamScores[bestIndex])
    
    history = [{
        "round": 0,
        "bestScore": bestScore,
        "beamBest": float(beamScores.max()),
        "beamMean": float(beamScores.mean()),
    }]

    noGainRounds = 0
    
    for roundNo in range(1,maxRounds+1):
        # gom vào 1 rổ chung
        neighborPool = list({
            newState
            for state in beam
            for newState in getNeighbors(state)
        })

        if not neighborPool:
            break
        
        neighborScores = np.array([objective(state, weights) for state in neighborPool])

        # đổi điểm thành số vé. điểm có thể âm nên dời tất cả lên dương
        # bằng cách trừ đi điểm thấp nhất. cộng baseTickets để phương án bét vẫn còn vé
        scoreSpread = neighborScores.max() - neighborScores.min()
        baseTickets = scoreSpread * 0.1 if scoreSpread > 0 else 1.0
        tickets = neighborScores - neighborScores.min() + baseTickets
        pickProbs = tickets / tickets.sum()
        
        # lấy beamWidth phương thức khác nhau
        pickedIndexes = rng.choice(len(neighborPool), size=beamWidth, replace=False, p=pickProbs)
        beam = [neighborPool[i] for i in pickedIndexes]
        beamScores = neighborScores[pickedIndexes]
        
        # dùng argMax để tìm phương án tốt nhất của vòng
        roundBestIndex = int(np.argmax(beamScores))
        roundBestScore = float(beamScores[roundBestIndex])
        
        if roundBestScore > bestScore:
            bestState = beam[roundBestIndex]
            bestScore = roundBestScore
            noGainRounds = 0
        else:
            noGainRounds += 1

        history.append({
            "round": roundNo,
            "bestScore": bestScore,
            "beamBest": roundBestScore,
            "beamMean": float(beamScores.mean()),
        })
        
        # thực hiện dùng nếu k có kết quả
        if noGainRounds >= maxNoGain:
            break
        
    return bestState, bestScore, history
        
        

## 4.3 randomRestartBeamSearch


In [5]:
def randomRestartBeamSearch(numRestarts, beamWidth, maxRounds=20, maxNoGain=10,
                            seed=0, numStations=None, weights=None,
                            searchFn=localBeamSearch):
    bestState = None
    bestScore = -np.inf
    history = []
    
    # Mỗi lần chạy phải đổi seed, không thì bốc trúng y hệt bộ phương án cũ
    # và chạy lại numRestarts lần cũng chỉ ra đúng một kết quả
    for restartNo  in range(numRestarts):
        state, score, runHistory = searchFn(
            beamWidth,
            maxRounds=maxRounds,
            maxNoGain=maxNoGain,
            seed=seed + restartNo,
            numStations=numStations,
            weights=weights,
        )
        
        # Gắn thêm số thứ tự lần chạy vào nhật ký, để Khối 6 tách ra vẽ riêng
        # từng lần và thấy được lần nào ăn may lần nào xui
        for row in runHistory:
            history.append({"restart": restartNo, **row})

        # Giữ lại phương án tốt nhất trong tất cả các lần chạy
        if score > bestScore:
            bestState = state
            bestScore = score

    return bestState, bestScore, history

# Khối 5. Thực nghiệm
Chạy các thuật toán ở Khối 4 trên dữ liệu thật, thu số liệu cho chương 4 báo cáo.

## 5.1 runExperiment — hàm chạy thí nghiệm dùng chung

Ba mục 5.2, 5.3, 5.4 thực chất là cùng một việc, chỉ khác thứ đem ra quét:

| Mục | Quét cái gì |
|---|---|
| 5.2 | `beamWidth` = 1, 3, 5, 10, 20 |
| 5.3 | Thuật toán = local beam, stochastic, random restart |
| 5.4 | Trọng số dân số / chi phí |


| Tên | Ý nghĩa |
|---|---|
| `searchFn` | Thuật toán đem chạy: `localBeamSearch`, `stochasticLocalBeamSearch`... |
| `numSeeds` | Chạy bao nhiêu lần, mặc định 20 |
| `label` | Tên cấu hình, để phân biệt các dòng trong bảng kết quả |
| `**params` | Tham số đưa thẳng cho `searchFn` (`beamWidth`, `maxRounds`, `weights`...) |

### Mỗi lần chạy ghi lại gì

| Cột | Ý nghĩa |
|---|---|
| `label` | Tên cấu hình |
| `seed` | Seed của lần chạy này |
| `score` | Điểm phương án tốt nhất tìm được |
| `numStations` | Phương án đó có mấy trạm |
| `rounds` | Chạy hết bao nhiêu vòng mới dừng |
| `seconds` | Thời gian chạy, dùng cho đồ thị đánh đổi ở Khối 6 |




In [7]:
import time

def runExperiment(algorithm,label, numSeeds=20,**params):

    rows = []

    for seed in range(numSeeds):
        start = time.perf_counter()
        bestState, bestScore, history = algorithm(seed=seed, **params)
        seconds = time.perf_counter() - start
        
        numRuns = len({row["restart"] for row in history}) if "restart" in history[0] else 1
        
        rows.append({
            "label": label,
                "seed": seed,
                "score": bestScore,
                "numStations": len(bestState),
                "rounds": len(history) - numRuns,
                "seconds": seconds,
        })
        
    return pd.DataFrame(rows)